In [5]:
# Check if the SEC filings dataset is available as local files in Kaggle
!ls -la /kaggle/input
!find /kaggle/input -maxdepth 4

total 8
drwxr-xr-x 2 root root 4096 Sep 10 18:31 .
drwxr-xr-x 5 root root 4096 Sep 10 18:31 ..


In [1]:
# No local files found. This dataset is BigQuery-only, so connect via the BigQuery client
# Lit all public datasets with "sec" in the name to confirm the right one
from google.cloud import bigquery
client = bigquery.Client(project="YOUR_PROJECT_ID")  # use the project ID shown after linking

for d in client.list_datasets(project="bigquery-public-data"):
    if "sec" in d.dataset_id.lower():
        print(d.dataset_id)

sec_quarterly_financials


In [5]:
# Set up the real BigQuery client using our linked GCP project
from google.cloud import bigquery
client = bigquery.Client(project="dsjungle-493315")
print(client.project)

dsjungle-493315


In [7]:
# Check the actual column names in the submission table since they don't match SEC's standard naming
query = """
SELECT column_name
FROM `bigquery-public-data.sec_quarterly_financials.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'submission'
"""
df = client.query(query).to_dataframe()
df

/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,column_name
0,submission_number
1,central_index_key
2,company_name
3,sic
4,countryba
5,stprba
6,cityba
7,zipba
8,ba_street1
9,ba_street2


In [8]:
# Show the merger filing types / check what filing types (forms) are in this dataset and how many of each
query = """
SELECT form, COUNT(DISTINCT submission_number) AS n
FROM `bigquery-public-data.sec_quarterly_financials.submission`
GROUP BY form
ORDER BY n DESC
"""
df = client.query(query).to_dataframe()
df

/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,form,n
0,10-Q,192496
1,10-K,59587
2,8-K,45471
3,10-Q/A,9707
4,10-K/A,3958
5,20-F,3615
6,S-1/A,2552
7,S-1,1849
8,6-K,1527
9,8-K/A,1060


In [9]:
# Get the tech-only count by filtering to merger-related filings (S-4, S-4/A, 425) for tech-related SIC codes
query = """
SELECT s.company_name, s.sic, s.form, s.date_filed
FROM `bigquery-public-data.sec_quarterly_financials.submission` s
WHERE s.form IN ('S-4', 'S-4/A', '425')
  AND s.sic IN ('7372','7370','7371','3674','3576','3577','4899','3661','3663','3669')
ORDER BY s.date_filed DESC
"""
df = client.query(query).to_dataframe()
print(len(df))
df

32


/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,company_name,sic,form,date_filed
0,SOUTH MOUNTAIN MERGER CORP.,7371,S-4/A,20201218
1,SOUTH MOUNTAIN MERGER CORP.,7371,S-4/A,20201125
2,"AMERI HOLDINGS, INC.",3674,S-4/A,20201110
3,"AMERI HOLDINGS, INC.",3674,S-4/A,20201109
4,"AMERI HOLDINGS, INC.",3674,S-4/A,20201022
5,TRINE ACQUISITION CORP.,3577,S-4/A,20201015
6,"PECK CO HOLDINGS, INC.",3674,S-4/A,20201014
7,"AMERI HOLDINGS, INC.",3674,S-4/A,20200917
8,DISH DBS CORP,4899,S-4,20200819
9,"AMERI HOLDINGS, INC.",3674,S-4/A,20200812


In [10]:
# Only 32 matches
# Check when the dataset was last updated
query = "SELECT MAX(date_filed) AS latest FROM `bigquery-public-data.sec_quarterly_financials.submission`"
client.query(query).to_dataframe(

/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,latest
0,20201231


In [13]:
# The one above only had ~15 useable findings through 2020
# Pulling from EDGAR instead since it covers filings from the 1990s through today


import requests
import xml.etree.ElementTree as ET
import pandas as pd
import time

headers = {"User-Agent": "Sydney Janssen sydneyjanssen9@gmail.com"}
sic_codes = ["7372","7370","7371","3674","3576","3577","4899","3661","3663","3669"]
forms = ["S-4", "425"]
ns = {"a": "http://www.w3.org/2005/Atom"}

rows = []
for sic in sic_codes:
    for form in forms:
        url = (f"https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany"
               f"&SIC={sic}&type={form}&dateb=&owner=include&count=100&output=atom")
        r = requests.get(url, headers=headers)
        if r.status_code != 200:
            continue
        try:
            root = ET.fromstring(r.content)
        except ET.ParseError:
            continue
        for entry in root.findall("a:entry", ns):
            rows.append({
                "sic": sic,
                "form": form,
                "title": entry.findtext("a:title", default="", namespaces=ns),
                "updated": entry.findtext("a:updated", default="", namespaces=ns),
                "link": (entry.find("a:link", ns).attrib.get("href")
                         if entry.find("a:link", ns) is not None else ""),
            })
        time.sleep(0.2)

df = pd.DataFrame(rows)
print(len(df))
df

1800


,sic,form,title,updated,link
0,7372,S-4,,2026-09-10T15:12:16-04:00,https://www.sec.gov/cgi-bin/browse-edgar?actio...
1,7372,S-4,,2026-09-10T15:12:16-04:00,https://www.sec.gov/cgi-bin/browse-edgar?actio...
2,7372,S-4,,2026-09-10T15:12:16-04:00,https://www.sec.gov/cgi-bin/browse-edgar?actio...
3,7372,S-4,,2026-09-10T15:12:16-04:00,https://www.sec.gov/cgi-bin/browse-edgar?actio...
4,7372,S-4,,2026-09-10T15:12:16-04:00,https://www.sec.gov/cgi-bin/browse-edgar?actio...
...,...,...,...,...,...
1795,3669,425,,2026-09-10T15:15:56-04:00,https://www.sec.gov/cgi-bin/browse-edgar?actio...
1796,3669,425,,2026-09-10T15:15:56-04:00,https://www.sec.gov/cgi-bin/browse-edgar?actio...
1797,3669,425,,2026-09-10T15:15:56-04:00,https://www.sec.gov/cgi-bin/browse-edgar?actio...
1798,3669,425,,2026-09-10T15:15:56-04:00,https://www.sec.gov/cgi-bin/browse-edgar?actio...


In [15]:
# Pull CIK/SIC fields to get a count of tech companies with an S-4 or 425 filing on record
rows = []
for sic in sic_codes:
    for form in forms:
        url = (f"https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany"
               f"&SIC={sic}&type={form}&dateb=&owner=include&count=100&output=atom")
        r = requests.get(url, headers=headers)
        if r.status_code != 200:
            continue
        try:
            root = ET.fromstring(r.content)
        except ET.ParseError:
            continue
        for entry in root.findall("a:entry", ns):
            content = entry.find("a:content", ns)
            ci = content.find("a:company-info", ns) if content is not None else None
            if ci is None:
                continue
            rows.append({
                "cik": ci.findtext("a:cik", default="", namespaces=ns),
                "sic": ci.findtext("a:sic", default="", namespaces=ns),
                "state": ci.findtext("a:state", default="", namespaces=ns),
                "form": form,
            })
        time.sleep(0.2)

df = pd.DataFrame(rows).drop_duplicates(subset=["cik"])
print(len(df))
df.head(20)

1000


,cik,sic,state,form
0,0001877461,7372,AZ,S-4
1,0001459417,7372,MD,S-4
2,0001556753,7372,C3,S-4
3,0000910638,7372,SC,S-4
4,0001023748,7372,CA,S-4
5,0001010026,7372,CA,S-4
6,0000898441,7372,CA,S-4
7,0001097641,7372,A6,S-4
8,0001971975,7372,A6,S-4
9,0000898739,7372,NJ,S-4


In [16]:
# Turn the CIK list above into real company names and actual filing dates 
# Using SEC's cleaner JSON API to get one call per unique company

import json

results = []
for cik in df['cik'].unique():
    cik_padded = cik.zfill(10)
    url = f"https://data.sec.gov/submissions/CIK{cik_padded}.json"
    r = requests.get(url, headers=headers)
    if r.status_code != 200:
        continue
    data = r.json()
    name = data.get("name", "")
    sic = data.get("sic", "")
    recent = data.get("filings", {}).get("recent", {})
    forms = recent.get("form", [])
    dates = recent.get("filingDate", [])
    for f, d in zip(forms, dates):
        if f in ("S-4", "S-4/A", "425"):
            results.append({"cik": cik, "company_name": name, "sic": sic, "form": f, "date_filed": d})
    time.sleep(0.15)

deals = pd.DataFrame(results)
print(len(deals))
deals.sort_values("date_filed", ascending=False)

3902


,cik,company_name,sic,form,date_filed
1162,0001730168,Broadcom Inc.,3674,S-4,2026-06-09
1161,0001730168,Broadcom Inc.,3674,S-4,2026-06-09
1921,0001590496,Aerkomm Inc.,4899,S-4,2026-01-16
422,0002034400,Black Titan Corp,7371,425,2025-10-03
1889,0001327567,Palo Alto Networks Inc,3577,S-4/A,2025-09-26
...,...,...,...,...,...
3199,0000016357,AB LIQUIDATING CORP,3663,S-4/A,1995-03-14
3508,0000808918,"EDCI HOLDINGS, INC.",3663,S-4/A,1995-03-09
3200,0000016357,AB LIQUIDATING CORP,3663,S-4/A,1995-02-14
3201,0000016357,AB LIQUIDATING CORP,3663,S-4,1995-02-03


In [18]:
# Save the output
deals.to_csv('/kaggle/working/tech_merger_filings.csv', index=False)

In [19]:
#3902 filings produced above, but mergers can show up multiple times so grouping by name + a rough time window

deals_sorted = deals.sort_values(['cik','date_filed']).reset_index(drop=True)
deals_sorted['date_filed'] = pd.to_datetime(deals_sorted['date_filed'])

deal_id = 0
deal_ids = []
prev_cik = None
prev_date = None
for _, row in deals_sorted.iterrows():
    if row['cik'] != prev_cik or (row['date_filed'] - prev_date).days > 400:
        deal_id += 1
    deal_ids.append(deal_id)
    prev_cik = row['cik']
    prev_date = row['date_filed']

deals_sorted['deal_id'] = deal_ids

deal_summary = deals_sorted.groupby('deal_id').agg(
    cik=('cik','first'),
    company_name=('company_name','first'),
    sic=('sic','first'),
    first_filed=('date_filed','min'),
    last_filed=('date_filed','max'),
    filings=('form', lambda x: ','.join(x)),
    n_filings=('form','count')
).reset_index()

print(len(deal_summary))
deal_summary.sort_values('first_filed', ascending=False)

347


,deal_id,cik,company_name,sic,first_filed,last_filed,filings,n_filings
309,310,0001590496,Aerkomm Inc.,4899,2026-01-16,2026-01-16,S-4,1
345,346,0002034400,Black Titan Corp,7371,2025-10-03,2025-10-03,425,1
324,325,0001730168,Broadcom Inc.,3674,2025-09-10,2026-06-09,"S-4,S-4,S-4,S-4",4
282,283,0001327567,Palo Alto Networks Inc,3577,2025-07-30,2025-09-26,"425,425,425,425,425,425,425,425,425,425,425,42...",16
311,312,0001630212,Change Agents Corporation.,7371,2025-03-10,2025-09-08,"425,S-4,425,S-4/A,425,425,425,425",8
...,...,...,...,...,...,...,...,...
53,54,0000804888,ACCLAIM ENTERTAINMENT INC,7372,1995-04-27,1995-06-20,"S-4,S-4/A,S-4/A",3
41,42,0000738076,3COM CORP,3576,1995-03-23,1995-08-31,"S-4,S-4/A,S-4/A,S-4/A,S-4/A,S-4",6
55,56,0000808918,"EDCI HOLDINGS, INC.",3663,1995-03-09,1995-03-24,"S-4/A,S-4/A",2
8,9,0000016357,AB LIQUIDATING CORP,3663,1995-02-03,1995-05-01,"S-4,S-4/A,S-4/A,S-4/A,S-4/A",5


In [21]:
# That returned 347 deals
# Save that deal summary table as a CSV to use as our identified dataset
deal_summary.to_csv('/kaggle/working/tech_merger_deals.csv', index=False)

In [22]:
!ls -la /kaggle/working

total 248
drwxr-xr-x 3 root root   4096 Sep 10 19:41 .
drwxr-xr-x 5 root root   4096 Sep 10 18:44 ..
-rw-r--r-- 1 root root  39298 Sep 10 19:41 tech_merger_deals.csv
-rw-r--r-- 1 root root 198647 Sep 10 19:36 tech_merger_filings.csv
drwxr-xr-x 2 root root   4096 Sep 10 18:44 .virtual_documents
